# 흉부 X-ray 폐렴 분류 최종 제출 코드

**최종 모델:** ImageNet 사전학습 ResNet18 + 학습 데이터 증강(RandomRotation 7°)

| 단계 | 내용 |
|---|---|
| 1 | 데이터 로드 및 검증 (파일명 일치, 클래스 비율, 이미지 크기/모드, Train/Validation 층화 분할) |
| 2 | 이미지 전처리 및 증강 |
| 3 | ResNet18 학습 및 검증 (검증 정확도 기준 최고 모델 저장) |
| 4 | 성능 평가 (Accuracy, Precision, Recall, F1, Confusion Matrix, 판정 기준 비교) |
| 5 | 테스트 예측 및 제출 CSV 생성 |
| 부록 | 실험 기록 (mission_3, mission_4) |

## 0. 환경 설정

In [ ]:
import os
import random
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# ---------------- 설정값 ----------------
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
LEARNING_RATE = 0.0001
NUM_EPOCHS    = 5
THRESHOLD     = 0.3      # 폐렴(1)으로 판정할 확률 기준 (기본 argmax는 0.5와 동일)

DATA_DIR      = '/content/chest_xray'
CKPT_PATH     = '/content/best_resnet18_aug.pth'
SUBMIT_PATH   = '/content/submission_resnet18_aug_threshold03.csv'

# mission_1·2와 mission_3·4에서 폴더명이 달랐으므로 존재하는 경로를 자동으로 선택
ZIP_CANDIDATES = [
    '/content/drive/MyDrive/오즈코딩/13_해커톤/open.zip',
    '/content/drive/MyDrive/오즈코딩/13해커톤/open.zip',
]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)

## 1. 데이터 로드 및 확인

In [ ]:
# 드라이브 마운트 & 압축 해제 (이미 풀려 있으면 건너뜀)
if not os.path.exists(f'{DATA_DIR}/train.csv'):
    from google.colab import drive
    drive.mount('/content/drive/')

    zip_path = next((p for p in ZIP_CANDIDATES if os.path.exists(p)), None)
    assert zip_path is not None, f'open.zip을 찾을 수 없습니다: {ZIP_CANDIDATES}'

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATA_DIR)
    print('압축 해제 완료:', zip_path)
else:
    print('이미 압축이 해제되어 있습니다.')

In [ ]:
# CSV 로드
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print('train:', train_df.shape, '/ test:', test_df.shape, '/ sample_submission:', sample_submission.shape)
print('train 컬럼:', train_df.columns.tolist())
print('결측치(train):', train_df.isnull().sum().to_dict(), '/ 결측치(test):', test_df.isnull().sum().to_dict())

In [ ]:
# CSV 파일명과 실제 이미지 파일명 일치 확인
for name, df in [('train', train_df), ('test', test_df)]:
    files_in_dir = set(os.listdir(f'{DATA_DIR}/{name}'))
    print(f'{name}: CSV {len(df)}행 / 이미지 {len(files_in_dir)}장 / 파일명 일치 {set(df["file_name"]) == files_in_dir}')

# 제출 양식의 파일명과 test.csv의 파일명 일치 확인 (순서는 달라도 되며, 제출 시 파일명 기준으로 매칭)
print('sample_submission과 test.csv 파일명 일치:', set(sample_submission['file_name']) == set(test_df['file_name']))

In [ ]:
# 클래스 분포
dist = train_df['label'].value_counts().sort_index()
print(pd.DataFrame({'count': dist, 'ratio(%)': (dist / dist.sum() * 100).round(2)}))

# 이미지 크기 및 모드 확인
size_counts, mode_counts = Counter(), Counter()
for file in train_df['file_name']:
    with Image.open(os.path.join(DATA_DIR, 'train', file)) as img:
        size_counts[img.size] += 1
        mode_counts[img.mode] += 1
print('이미지 크기별 개수:', size_counts)
print('이미지 모드별 개수:', mode_counts)

In [ ]:
# Train / Validation 층화 분할 (label 비율 유지)
train_split, val_split = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df['label']
)
print(f'Train: {len(train_split)} / Validation: {len(val_split)} / Test: {len(test_df)}')
print('Train label 비율:', train_split['label'].value_counts(normalize=True).round(4).to_dict())
print('Val   label 비율:', val_split['label'].value_counts(normalize=True).round(4).to_dict())

## 2. 이미지 전처리 및 증강

- 이미지 크기 224 × 224, grayscale → 3채널 변환, ImageNet 평균·표준편차로 정규화
- **학습 데이터에만** `RandomRotation(7)` 적용 (검증·테스트는 증강 없음)

In [ ]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(7),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    normalize,
])

eval_transform = transforms.Compose([      # 검증·테스트 공통
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    normalize,
])

class ChestXrayDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.has_label = 'label' in self.df.columns

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.image_dir, row['file_name'])).convert('L')
        if self.transform:
            image = self.transform(image)
        if self.has_label:
            return image, int(row['label'])
        return image

train_dir, test_dir = f'{DATA_DIR}/train', f'{DATA_DIR}/test'

train_dataset = ChestXrayDataset(train_split, train_dir, train_transform)
val_dataset   = ChestXrayDataset(val_split,   train_dir, eval_transform)
test_dataset  = ChestXrayDataset(test_df,     test_dir,  eval_transform)

g = torch.Generator()
g.manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

images, labels = next(iter(train_loader))
print('이미지 형태:', images.shape, '/ 레이블 형태:', labels.shape)

## 3. 모델 학습 및 검증

- ImageNet 사전학습 ResNet18, 마지막 분류층을 2개 클래스(정상 / 폐렴)로 교체
- Adam, 학습률 0.0001, 5 Epoch
- **검증 정확도가 가장 높은 Epoch의 가중치를 저장**하고, 학습 후 해당 가중치로 평가·예측 (동률이면 검증 Loss가 낮은 쪽)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc, best_val_loss, best_epoch = 0.0, float('inf'), 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    for k, v in zip(history.keys(), [train_loss, train_acc, val_loss, val_acc]):
        history[k].append(v)

    improved = (val_acc > best_val_acc) or (val_acc == best_val_acc and val_loss < best_val_loss)
    if improved:
        best_val_acc, best_val_loss, best_epoch = val_acc, val_loss, epoch
        torch.save(model.state_dict(), CKPT_PATH)

    print(f'Epoch [{epoch}/{NUM_EPOCHS}] Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}' + ('  <- best 저장' if improved else ''))

print(f'\n최고 성능 Epoch: {best_epoch} / 검증 정확도: {best_val_acc:.4f} / 검증 Loss: {best_val_loss:.4f}')

In [ ]:
# 학습 곡선
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history['train_loss'], marker='o', label='Train')
axes[0].plot(epochs, history['val_loss'], marker='o', label='Validation')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(epochs, history['train_acc'], marker='o', label='Train')
axes[1].plot(epochs, history['val_acc'], marker='o', label='Validation')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

## 4. 성능 평가

In [ ]:
# 저장해 둔 최고 성능 모델 불러오기
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model = model.to(device)
model.eval()

def predict_probs(loader):
    probs = []
    with torch.no_grad():
        for batch in loader:
            images = batch[0] if isinstance(batch, (list, tuple)) else batch
            outputs = model(images.to(device))
            probs.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())
    return np.array(probs)

val_probs = predict_probs(val_loader)
val_labels = val_split['label'].values   # val_loader는 shuffle=False이므로 val_split 순서와 동일
print('검증 샘플 수:', len(val_probs))

In [ ]:
# 판정 기준(threshold)별 성능 비교 - 폐렴(1)을 양성으로 계산
rows = []
for th in [0.5, 0.4, 0.3, 0.2]:
    preds = (val_probs >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(val_labels, preds).ravel()
    rows.append({
        'threshold': th,
        'Accuracy': accuracy_score(val_labels, preds),
        'Precision': precision_score(val_labels, preds),
        'Recall': recall_score(val_labels, preds),
        'F1': f1_score(val_labels, preds),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
    })
threshold_df = pd.DataFrame(rows).set_index('threshold').round(4)
threshold_df

In [ ]:
# 최종 판정 기준(THRESHOLD)의 상세 평가
val_preds = (val_probs >= THRESHOLD).astype(int)

print(f'판정 기준: {THRESHOLD}')
print(classification_report(val_labels, val_preds, target_names=['Normal', 'Pneumonia'], digits=4))

cm = confusion_matrix(val_labels, val_preds)
print('Confusion Matrix:\n', cm)
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Pneumonia']).plot(cmap='Blues', values_format='d')
plt.title(f'Validation Confusion Matrix (threshold={THRESHOLD})')
plt.show()

## 5. 테스트 예측 및 제출 파일 생성

테스트 이미지의 폐렴 확률을 예측하고, `sample_submission.csv`의 **파일명 순서에 맞춰** 결과를 연결합니다.
(`test.csv`와 `sample_submission.csv`의 행 순서가 같다고 가정하지 않고 `file_name`으로 매칭)

In [ ]:
test_probs = predict_probs(test_loader)
test_preds = (test_probs >= THRESHOLD).astype(int)

pred_df = pd.DataFrame({'file_name': test_df['file_name'].values, 'label': test_preds})

submission = sample_submission[['file_name']].copy()
submission['label'] = submission['file_name'].map(pred_df.set_index('file_name')['label'])

# 검증
assert len(submission) == len(test_df) == len(test_probs)
assert submission['file_name'].is_unique
assert submission['label'].notna().all(), '누락된 예측값이 있습니다.'
assert set(submission['label'].unique()).issubset({0, 1})

submission['label'] = submission['label'].astype(int)
print('제출 데이터 크기:', submission.shape)
print('예측 클래스별 개수:')
print(submission['label'].value_counts().sort_index())
print('결측치:', submission['label'].isnull().sum())
submission.head()

In [ ]:
submission.to_csv(SUBMIT_PATH, index=False)
print('CSV 저장 완료:', SUBMIT_PATH)

try:
    from google.colab import files as colab_files
    colab_files.download(SUBMIT_PATH)
except ImportError:
    pass

## 부록. 실험 기록

### 모델별 성능 비교

| 실험 | 모델 | 증강 | 학습률 | Epoch | 검증 정확도 | 제출 점수 |
|---|---|---|---|---|---|---|
| mission_3 | ResNet18 | 없음 | 0.001 | 5 | 98.56% | 0.91100 또는 0.91185 (확인 필요) |
| mission_4 | ResNet18 | RandomRotation(7) | 0.0001 | 5 | 99.04% (Epoch 2 기준) | **0.93589** (판정 기준 0.3) |
| mission_4 | DenseNet121 | RandomRotation(7) | 0.0001 | 5 | 99.43% (확인 필요) | 0.91346 |

- 최종 모델은 제출 점수가 가장 높았던 **ResNet18 + 데이터 증강**입니다.
- DenseNet121은 검증 정확도가 더 높았지만 제출 점수는 낮아, 검증 성능과 테스트 성능이 항상 일치하지는 않음을 확인했습니다.

### DenseNet121 판정 기준 비교 (mission_4, Validation)

| 판정 기준 | Accuracy | Pneumonia Recall | 혼동 행렬 [[TN, FP], [FN, TP]] |
|---|---|---|---|
| 0.4 | 0.9875 | 0.9858 | [[266, 2], [11, 765]] |
| 0.3 | 0.9885 | 0.9871 | [[266, 2], [10, 766]] |
| 0.2 | 0.9875 | 0.9897 | [[263, 5], [8, 768]] |

### 최종 코드와 기존 노트북의 차이

- 최종 코드는 검증 정확도가 가장 높은 Epoch의 가중치를 저장해 사용합니다. 기존 mission_4의 ResNet18 실험은 마지막(5번째) Epoch 모델(검증 정확도 98.95%)을 그대로 사용했습니다.
- 판정 기준 0.3은 DenseNet121 검증 결과로 비교한 뒤 ResNet18 제출에도 적용했습니다. 최종 코드에서는 ResNet18 자체의 검증 확률로 판정 기준을 다시 비교합니다 (4단계 표 참고).
- 재현성을 위해 random seed(42)를 고정했습니다. 기존 실험은 seed를 고정하지 않아 Epoch별 수치가 다시 실행하면 달라질 수 있습니다.